In [1]:
import requests
import datetime
import statsapi
import pandas as pd

In [ ]:
# Using mlb-statsapi to get mlb schedule up to the all-star break
# Create dataframe from list genereated from api call
# Creating mapping of mlb team name -> team id

sched = statsapi.schedule(start_date='04/01/2024',end_date='07/16/2024')
sched_df = pd.DataFrame(sched)
teams = sched_df[['away_id', 'away_name']]
teams = teams.drop_duplicates(subset=['away_id'], keep='first')
teams = teams.sort_values(by='away_name', ascending=True)
teams = teams.rename(columns={'away_name': 'team', 'away_id': 'id'})



In [4]:
# MLB Schedule for current date

def print_schedule(schedule):
    """Prints the MLB schedule.

    Args:
        schedule (list): MLB schedule from api call.
    """
    for games in schedule:
        print(f"Date: {games['game_date']} | {games['away_name']} ({games['away_probable_pitcher']}) @ {games["home_name"]} ({games['home_probable_pitcher']})\n")

# {get_pitcher_stats(games['away_probable_pitcher'])}


today = datetime.date.today().strftime('%m/%d/%Y')
mlb_schedule = statsapi.schedule(start_date=today, end_date=today)

print(mlb_schedule)

[{'game_id': 823649, 'game_datetime': '2026-03-26T17:15:00Z', 'game_date': '2026-03-26', 'game_type': 'R', 'status': 'In Progress', 'away_name': 'Pittsburgh Pirates', 'home_name': 'New York Mets', 'away_id': 134, 'home_id': 121, 'doubleheader': 'N', 'game_num': 1, 'home_probable_pitcher': 'Freddy Peralta', 'away_probable_pitcher': 'Paul Skenes', 'home_pitcher_note': '', 'away_pitcher_note': '', 'away_score': 3, 'home_score': 5, 'current_inning': 3, 'inning_state': 'Bottom', 'venue_id': 3289, 'venue_name': 'Citi Field', 'national_broadcasts': ['NBC/Peacock'], 'series_status': None, 'summary': '2026-03-26 - Pittsburgh Pirates (3) @ New York Mets (5) (Bottom of the 3rd)'}, {'game_id': 823812, 'game_datetime': '2026-03-26T18:10:00Z', 'game_date': '2026-03-26', 'game_type': 'R', 'status': 'In Progress', 'away_name': 'Chicago White Sox', 'home_name': 'Milwaukee Brewers', 'away_id': 145, 'home_id': 158, 'doubleheader': 'N', 'game_num': 1, 'home_probable_pitcher': 'Jacob Misiorowski', 'away_pr

In [37]:
import re
import json

# Get pitcher names from MLB schedule
# Store away/home pitcher names into dictionary
# Get pitcher stats and store into another dictionary

def get_pitcher_names(schedule):
    """Gets pitcher names from MLB.

    Args:
        schedule (list): MLB schedule for current day.

    Returns:
        list: Away/Home pitcher names.
    """

    names = []
    for games in schedule:
        names.append({"away": games["away_probable_pitcher"], "home": games["home_probable_pitcher"]})
    return names



def get_pitcher_stats(name):
    """Gets pitcher stats for current year.

    Args:
        name (str): Pitcher full name.

    Returns:
        str: String of all pitcher stats.
    """
    try:
        pitcher = statsapi.player_stats(next(x['id'] for x in statsapi.get('sports_players', {'season':2024})['people'] if x['fullName']==name), 'pitching')
    except StopIteration:
        return ""
        
    return pitcher


def store_pitcher_stats(name, pitcher):
    """Get pitcher stats for current season.

    Args:
        name (str): Pitcher name.
        pitcher (list): List of pitcher stats

    Returns:
        dictionary: for pitcher stats
    """
    
    current_pitcher = {}
    
    stats_pattern = r"(\w+):\s([^\n]+)"
    stats = re.findall(stats_pattern, pitcher)
    
    current_pitcher[name] = dict(stats)
    
    return current_pitcher

def print_pitcher_stats(names):
    
    
    
    raise NotImplementedError

def create_pitcher_df(pitchers):
    pitcher_stats = {}
    for pitcher in pitchers:
        # print(pitcher)
        away = get_pitcher_stats(pitcher["away"])
        home = get_pitcher_stats(pitcher["home"])
    
        away_stats = store_pitcher_stats(pitcher["away"],away)
        home_stats = store_pitcher_stats(pitcher["home"],home)
        # print(f"{away_stats}\t{home_stats}")
        pitcher_stats.update(away_stats)
        pitcher_stats.update(home_stats)
    df = pd.DataFrame(pitcher_stats).transpose()
    df.fillna(0, inplace=True)
    return df

pitcher = statsapi.player_stats(next(x['id'] for x in statsapi.get('sports_players', {'season':2024})['people'] if x['fullName']=='Hunter Greene'), 'pitching')

pitchers = get_pitcher_names(mlb_schedule)
pitcher_df = create_pitcher_df(pitchers)


In [6]:
# 103 AL 104 NL
print(statsapi.standings_data(103))
print(statsapi.standings_data(104))

{201: {'div_name': 'American League East', 'teams': [{'name': 'New York Yankees', 'div_rank': '1', 'w': 1, 'l': 0, 'gb': '-', 'wc_rank': '-', 'wc_gb': '-', 'wc_elim_num': '-', 'elim_num': '-', 'team_id': 147, 'league_rank': '1', 'sport_rank': '1'}, {'name': 'Toronto Blue Jays', 'div_rank': '2', 'w': 0, 'l': 0, 'gb': '0.5', 'wc_rank': '1', 'wc_gb': '-', 'wc_elim_num': '-', 'elim_num': '-', 'team_id': 141, 'league_rank': '5', 'sport_rank': '5'}, {'name': 'Baltimore Orioles', 'div_rank': '3', 'w': 0, 'l': 0, 'gb': '0.5', 'wc_rank': '2', 'wc_gb': '-', 'wc_elim_num': '-', 'elim_num': '-', 'team_id': 110, 'league_rank': '8', 'sport_rank': '8'}, {'name': 'Tampa Bay Rays', 'div_rank': '4', 'w': 0, 'l': 0, 'gb': '0.5', 'wc_rank': '3', 'wc_gb': '-', 'wc_elim_num': '-', 'elim_num': '-', 'team_id': 139, 'league_rank': '10', 'sport_rank': '10'}, {'name': 'Boston Red Sox', 'div_rank': '5', 'w': 0, 'l': 0, 'gb': '0.5', 'wc_rank': '4', 'wc_gb': '-', 'wc_elim_num': '-', 'elim_num': '-', 'team_id': 111,

In [5]:
# https://github.com/toddrob99/MLB-StatsAPI/wiki/Function:-meta
# Get available values from StatsAPI for use in other queries, or look up descriptions for values found in API results.

meta = statsapi.meta('statGroups')


# https://github.com/toddrob99/MLB-StatsAPI/wiki/Function:-notes
# Get notes for a given endpoint.

notes = statsapi.notes('person_stats')
print(notes)

Endpoint: person_stats 
All path parameters: ['ver', 'personId', 'gamePk']. 
Required path parameters (note: ver will be included by default): ['ver', 'personId', 'gamePk']. 
All query parameters: ['fields']. 
Required query parameters: None. 
Developer notes: Specify "current" instead of a gamePk for a player's current game stats.
